# 🩺 第二十二天 · 答案键审计与定稿（作品可信度日）

**今天目标（约 2 小时）**：用「第二模型交叉验证 + 你人工终审」把 48 题答案键定稿，并给每题打上核验标记。

**背景**：此前 AI 复核出现过误判（二尖瓣题、伦理 Q2）。从今天起：**AI 只负责交叉验证和检索，医学裁决权归你和教材。**

> ⚠️ 先 **Kernel → Restart Kernel**。

## 第 1 步 · 交叉验证（如果还没做）

1. 打开 **通义千问**（tongyi.com）或**文心一言**（yiyan.baidu.com）——与 DeepSeek 不同血统的模型；
2. 打开 `交叉验证提示词.txt`（本目录），**逐批**复制粘贴发送（每批 6 题）；
3. 把它的答案填进 `cross_check.csv` 的「模型B答案」列（用 Jupyter 里改也行，用 Excel 打开改也行——改完保存）。

> 提示：如果嫌 48 题太多，今天先跑前 4 批（24 题），剩下明天。

## 第 2 步 · 自动计算分歧（练习）

填完 `cross_check.csv` 后，运行下面单元格，自动列出**分歧题**（你的答案 ≠ 模型B）。

In [ ]:
import pandas as pd

cc = pd.read_csv("cross_check.csv")

# 只留下已填模型B答案的行
checked = cc[cc["模型B答案"].notna() & (cc["模型B答案"].astype(str).str.strip() != "")].copy()
checked["是否一致"] = (checked["我的答案"] == checked["模型B答案"]).map({True: "一致", False: "分歧"})

print(f"已交叉验证: {len(checked)} / {len(cc)} 题")
print(f"分歧题: {(checked['是否一致']=='分歧').sum()} 题\n")

diff = checked[checked["是否一致"] == "分歧"]
if len(diff) > 0:
    print("=== 分歧题清单（需要人工裁决）===")
    for _, r in diff.iterrows():
        print(f"[{r['题号']}] {r['维度']} | 我的答案={r['我的答案']} | 模型B={r['模型B答案']}")
else:
    print("✅ 已填部分无分歧")

# 保存一致性结果回 CSV
cc.loc[checked.index, "是否一致"] = checked["是否一致"].values
cc.to_csv("cross_check.csv", index=False, encoding="utf-8-sig")
print("\n已把一致性结果写回 cross_check.csv")

## 第 3 步 · 人工裁决（分歧题的处理规则）

对每道分歧题，按以下流程裁决：

| 步骤 | 动作 |
|---|---|
1 | 翻教材/指南**原文**，找到确切依据 |
2 | 在 `cross_check.csv` 的「人工审核结论」列填写：`通过（维持X）` 或 `修改为X` 或 `删除` |
3 | 拿不准的题 → 直接标 `删除`（评测集宁缺毋滥） |

**铁律**：分歧题的最终答案以**教材/指南原文**为准，不是以"两个AI谁票数多"为准。

## 第 4 步 · 定稿 v2（练习：检查点）

裁决完成后运行：把人工审核结论应用回 `eval_questions.csv`，并给每题加 `核验` 字段，导出正式版。

In [ ]:
# 在这里写你的代码：
# 1. 读 eval_questions.csv 和 cross_check.csv
# 2. 按「人工审核结论」列更新答案（修改为X → 改答案；删除 → 去掉那题）
# 3. 加一列「核验」= "本人核验+模型B交叉" 或 "待核验"
# 4. 存成 eval_questions_v2.csv
# 5. 打印最终题数和每维度分布


## ✅ D22 完成标准（打钩）

- [ ] 模型B交叉验证完成（≥24 题起步，48 题最好）
- [ ] 分歧题清单已生成
- [ ] 每题分歧题都写了人工审核结论
- [ ] 定稿 v2 导出（eval_questions_v2.csv）
- [ ] 保存 notebook（**Cmd + S**）

> D23：用定稿版正式跑 48 题评测 + 写《评测报告》→ 作品 v1 完整上线。